In [ ]:
import re
from collections import Counter

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
model_name = "textattack/bert-base-uncased-MRPC"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()

print(f"Loaded model: {model_name}")
print(f"Number of labels: {model.config.num_labels}")

In [ ]:
dataset = load_dataset("glue", "mrpc", split="validation")
print("Dataset split: glue/mrpc validation")
print(f"Number of examples: {len(dataset)}")
print("Example row:")
print(dataset[0])

In [ ]:
token_pattern = re.compile(r"\b\w+\b")

def tokenize_to_word_set(text):
    return set(token_pattern.findall(text.lower()))

def jaccard_overlap(a, b):
    if not a and not b:
        return 1.0
    union = a | b
    if not union:
        return 0.0
    return len(a & b) / len(union)

rows = []
overlaps = []

for i, row in enumerate(dataset):
    s1_words = tokenize_to_word_set(row["sentence1"])
    s2_words = tokenize_to_word_set(row["sentence2"])
    overlap = jaccard_overlap(s1_words, s2_words)
    overlaps.append(overlap)
    rows.append({
        "idx": i,
        "sentence1": row["sentence1"],
        "sentence2": row["sentence2"],
        "label": row["label"],
        "overlap": overlap
    })

sorted_overlaps = sorted(overlaps)
n = len(sorted_overlaps)
low_overlap_threshold = sorted_overlaps[n // 4]
high_overlap_threshold = sorted_overlaps[(3 * n) // 4]

hard_examples = []
for item in rows:
    category = None
    if item["label"] == 0 and item["overlap"] >= high_overlap_threshold:
        category = "high_overlap_negative"
    elif item["label"] == 1 and item["overlap"] <= low_overlap_threshold:
        category = "low_overlap_positive"
    if category is not None:
        hard_examples.append({**item, "hard_category": category})

category_counts = Counter(x["hard_category"] for x in hard_examples)

print(f"Low-overlap threshold  (bottom quartile): <= {low_overlap_threshold:.4f}")
print(f"High-overlap threshold (top quartile)   : >= {high_overlap_threshold:.4f}")
print(f"Hard subset size: {len(hard_examples)}")
print("Hard subset category counts:")
print(dict(category_counts))
print("Sample hard example:")
print(hard_examples[0] if hard_examples else "No hard examples found.")

In [ ]:
batch_size = 32
predictions = []
confidences = []
true_labels = [x["label"] for x in hard_examples]

for start_idx in range(0, len(hard_examples), batch_size):
    batch = hard_examples[start_idx:start_idx + batch_size]
    inputs = tokenizer(
        [x["sentence1"] for x in batch],
        [x["sentence2"] for x in batch],
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        preds = torch.argmax(logits, dim=-1)
    predictions.extend(preds.cpu().tolist())
    confidences.extend(probs.max(dim=-1).values.cpu().tolist())

for i, pred in enumerate(predictions):
    hard_examples[i]["pred_label"] = pred
    hard_examples[i]["confidence"] = confidences[i]

print(f"Completed inference for {len(predictions)} hard-subset examples.")

In [ ]:
accuracy = accuracy_score(true_labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predictions, average="binary", zero_division=0)
cm = confusion_matrix(true_labels, predictions)

print("Overall hard-subset evaluation metrics:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")
print("Confusion matrix:")
print(cm)

In [ ]:
category_metrics = {}
categories = ["high_overlap_negative", "low_overlap_positive"]

for category in categories:
    subset = [x for x in hard_examples if x["hard_category"] == category]
    y_true = [x["label"] for x in subset]
    y_pred = [x["pred_label"] for x in subset]
    category_accuracy = accuracy_score(y_true, y_pred)
    category_precision, category_recall, category_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    category_cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    category_metrics[category] = {
        "count": len(subset),
        "accuracy": category_accuracy,
        "precision": category_precision,
        "recall": category_recall,
        "f1": category_f1,
        "confusion_matrix": category_cm.tolist()
    }

print("Hard-category metrics:")
for category in categories:
    m = category_metrics[category]
    print(f"category={category} count={m['count']} accuracy={m['accuracy']:.4f} precision={m['precision']:.4f} recall={m['recall']:.4f} f1={m['f1']:.4f}")
    print(f"confusion_matrix={m['confusion_matrix']}")

In [ ]:
label_map = {0: "not_paraphrase", 1: "paraphrase"}

for category in ["high_overlap_negative", "low_overlap_positive"]:
    failures = [x for x in hard_examples if x["hard_category"] == category and x["label"] != x["pred_label"]]
    failures = sorted(failures, key=lambda x: (-x["confidence"], -x["overlap"]))
    num_examples_to_show = min(5, len(failures))
    print(f"Most confident failures for category={category}: showing {num_examples_to_show} of {len(failures)}")
    for item in failures[:num_examples_to_show]:
        print(f"Original index: {item['idx']}")
        print(f"Category: {item['hard_category']}")
        print(f"Lexical overlap (Jaccard): {item['overlap']:.4f}")
        print(f"sentence1: {item['sentence1']}")
        print(f"sentence2: {item['sentence2']}")
        print(f"true label: {item['label']} ({label_map[item['label']]})")
        print(f"pred label: {item['pred_label']} ({label_map[item['pred_label']]})")
        print(f"confidence: {item['confidence']:.4f}")
        print("-" * 80)

In [ ]:
print("RESULT SUMMARY")
print(f"model={model_name}")
print("dataset_split=glue/mrpc validation")
print("subset_definition=hard_cases_from_lexical_overlap")
print(f"device={device}")
print(f"num_validation_examples={len(dataset)}")
print(f"num_hard_subset_examples={len(hard_examples)}")
print(f"low_overlap_threshold={low_overlap_threshold:.4f}")
print(f"high_overlap_threshold={high_overlap_threshold:.4f}")
print(f"overall_accuracy={accuracy:.4f}")
print(f"overall_precision={precision:.4f}")
print(f"overall_recall={recall:.4f}")
print(f"overall_f1={f1:.4f}")
for category in ["high_overlap_negative", "low_overlap_positive"]:
    m = category_metrics[category]
    print(f"category_{category}_count={m['count']}")
    print(f"category_{category}_accuracy={m['accuracy']:.4f}")
    print(f"category_{category}_precision={m['precision']:.4f}")
    print(f"category_{category}_recall={m['recall']:.4f}")
    print(f"category_{category}_f1={m['f1']:.4f}")